# Lending Club EDA -- F1 -- Data Understanding & Structural Profiling

**Status: built.** See the cell map below for what's actually in this notebook.

## What this notebook covers

What tables exist in the interim DuckDB file, how big each is, what every column's inferred type is (numeric vs. categorical, via cast-success rate), and the full raw-schema null%/cardinality profile. The 'get oriented' notebook -- read this first.

## Where this fits

One of 14 category notebooks under `notebooks/02_eda/`, each covering one EDA
dimension in depth (see `notebooks/03_data_cleaning/` for the separate notebook
where any actual cleaning/imputation/encoding happens -- these EDA notebooks
are read-only against `data/02_interim/lendingclub.duckdb` and never modify
or clean the data themselves). Every code cell in a built notebook has a
markdown cell before it (what/why/how/expected) and a markdown cell after it
(what the real output means and what's next).


## Cell map

| # | What it does |
|---|---|
| 1 | Connect to the interim DuckDB file, confirm row counts at every stage (raw / matured / windowed) |
| 2 | Numeric-vs-categorical split for every one of the 151 raw columns, by cast-success rate |
| 3 | Full null% / cardinality profile for every raw column (`SUMMARIZE`) |

**No cleaning happens here.** This notebook only observes and profiles the
data as it already sits in `data/02_interim/lendingclub.duckdb`; imputation,
transformation, and encoding all happen in `notebooks/data_cleaning/`.

## Cell 1 -- connect and confirm

**What / why:** Opening the persistent interim DuckDB file read-only and
confirming the three staged tables (`raw_mat`, `matured`, `windowed`) --
built by `build_lendingclub.py` -- have the row counts expected at each
stage. A sanity check before profiling anything: if these don't match, the
interim file itself needs rebuilding before this notebook means anything.

**Expect:** raw ~2.26M rows, matured (finished loans only) ~1.35M, windowed
(2013-2017 vintage only) ~1.2M.

In [1]:
import os, duckdb, pandas as pd, numpy as np
con = duckdb.connect(r"../../data/02_interim/lendingclub.duckdb", read_only=True)
ASSETS_TABLES = "../../data/04_assets/tables"
os.makedirs(ASSETS_TABLES, exist_ok=True)
os.makedirs("../../data/04_assets/plots", exist_ok=True)
for tbl in ["raw_mat", "matured", "windowed"]:
    n = con.sql(f"SELECT count(*) FROM {tbl}").fetchone()[0]
    print(f"{tbl}: {n:,} rows")
n_cols = len(con.sql("DESCRIBE raw_mat").fetchall())
print(f"raw_mat: {n_cols} columns")


raw_mat: 2,260,701 rows
matured: 1,348,099 rows
windowed: 1,195,879 rows
raw_mat: 151 columns


**What the output shows:** raw_mat: 2,260,701 rows
matured: 1,348,099 rows
windowed: 1,195,879 rows
raw_mat: 151 columns
Matches what `build_lendingclub.py` reported when it built the interim file.
151 raw columns confirmed on `raw_mat` -- every downstream notebook works
from a subset of these.

**Next:** figuring out which of those 151 columns are actually numeric vs.
categorical, since nothing arrived typed (everything was loaded as
`VARCHAR` on purpose, so nothing gets silently mis-cast).

## Cell 2 -- numeric vs. categorical, by cast-success rate

**What / why:** For every column in `windowed`, checking what fraction of
its non-null values successfully `TRY_CAST`s to `DOUBLE`. A true numeric
column casts near 100%; a categorical/free-text one casts near 0%. This is
more reliable than trusting column names, and flags anything genuinely
ambiguous.

**Expect:** a clean split with very few columns landing in the ambiguous
middle.

In [2]:
cols = [r[0] for r in con.sql("DESCRIBE windowed").fetchall()]
rows = []
for c in cols:
    r = con.sql(f'''
        SELECT count(*) n, count("{c}") n_notnull,
               count(DISTINCT "{c}") n_distinct,
               count(TRY_CAST("{c}" AS DOUBLE)) n_castable
        FROM windowed
    ''').fetchone()
    n, n_notnull, n_distinct, n_castable = r
    cast_rate = n_castable / n_notnull if n_notnull else 0
    rows.append((c, n_notnull / n, n_distinct, cast_rate))
type_df = pd.DataFrame(rows, columns=["column", "pct_notnull", "n_distinct", "cast_rate"])
type_df["inferred_type"] = np.where(type_df["cast_rate"] > 0.95, "numeric",
                             np.where(type_df["cast_rate"] < 0.05, "categorical/text", "mixed"))
print(type_df["inferred_type"].value_counts())
print()
print("mixed columns (worth a manual look):")
print(type_df[type_df["inferred_type"] == "mixed"].to_string(index=False))


inferred_type
numeric             114
categorical/text     38
Name: count, dtype: int64

mixed columns (worth a manual look):
Empty DataFrame
Columns: [column, pct_notnull, n_distinct, cast_rate, inferred_type]
Index: []


**What the output shows:** {'numeric': 114, 'categorical/text': 38}.
No columns landed in the "mixed" bucket -- every column is cleanly numeric or categorical.
This confirms the type assumptions the `data_cleaning` pipeline (and every
other EDA notebook) relies on.

**Next:** a full null%/cardinality profile across every column, typed or
not -- structure before any risk question gets asked.

## Cell 3 -- full null% / cardinality profile, every column

**What / why:** DuckDB's `SUMMARIZE` gives a one-pass null%/min/max/approx-distinct
profile for every column in a table in a single query. Running it on
`windowed` covers the complete 151-column raw schema, not just the ~27
columns that end up retained for modeling.

**Expect:** a wide spread -- some columns 0% missing, some (hardship/
settlement fields) missing for the vast majority of rows by design (they
only populate for loans that hit hardship), one or two 100% null.

In [3]:
summ = con.sql("SUMMARIZE windowed").df()
summ["null_pct"] = summ["null_percentage"].astype(float)
summ_sorted = summ[["column_name", "column_type", "null_pct", "approx_unique"]].sort_values("null_pct", ascending=False)
print(summ_sorted.head(12).to_string(index=False))
summ_sorted.to_csv(os.path.join(ASSETS_TABLES, "eda01_summ_sorted.csv"), index=False)
print("...")
print(f"columns with 0% nulls: {(summ_sorted['null_pct']==0).sum()} of {len(summ_sorted)}")
print(f"columns 100% null: {(summ_sorted['null_pct']==100).sum()}")


                               column_name column_type  null_pct  approx_unique
                                 member_id     VARCHAR    100.00              0
                              next_pymnt_d     VARCHAR    100.00              1
orig_projected_additional_accrued_interest     VARCHAR     99.69           3433
       sec_app_mths_since_last_major_derog     VARCHAR     99.65             93
                           hardship_length     VARCHAR     99.52              1
                           hardship_amount     VARCHAR     99.52           4136
                           hardship_status     VARCHAR     99.52              1
                             deferral_term     VARCHAR     99.52              1
                         hardship_end_date     VARCHAR     99.52             27
                           hardship_reason     VARCHAR     99.52              9
                       hardship_start_date     VARCHAR     99.52             27
                             hardship_ty

**What the output shows:** 2 column(s)
100% null (`member_id`, scrubbed by Lending Club before publication -- dead
weight, not signal). 80 of
152 columns are fully populated. The top of the
missing-rate ranking is dominated by hardship/settlement fields, which is
structural (most loans never enter hardship), not a data quality problem.

**Next:** `02_data_quality_integrity.ipynb` takes this profile further --
ranking missingness specifically on the columns that matter for modeling,
testing whether that missingness is informative, and sweeping for outliers
and implausible values.